## 4.1 C4.5 决策树 - 核心逻辑&手算方法

#### 1. C4.5决策树的核心逻辑
C4.5决策树是ID3算法的改进版本，主要通过引入信息增益率来选择划分特征，从而避免了ID3算法中倾向于选择取值较多的特征的问题。<br>
C4.5算法的核心逻辑包括以下几个步骤：
1. **计算信息增益率**：对于每个特征，计算其信息增益率。信息增益率是信息增益与特征的固有值（intrinsic value）的比值。特征的固有值是特征取值的分布情况的度量，计算公式为：IV(A) = -∑(P(v) * log2(P(v)))，其中P(v)是特征A取值v的概率。
2. **选择最优特征**：选择信息增益率最高的特征作为划分特征。
3. **递归划分**：根据选择的特征将数据集划分成子集，并对每个子集重复上述步骤，直到满足停止条件（如所有样本属于同一类，或没有更多特征可以划分）。
4. **剪枝**：C4.5算法还引入了剪枝机制，通过评估子树的性能来决定是否剪枝，以防止过拟合。

#### 2. C4.5 信息增益率的计算方式
1. **计算信息增益**：首先计算每个特征的信息增益，方法与ID3算法相同。
2. **计算特征的固有值**：对于每个特征，计算其固有值，衡量特征取值的分布情况。
 - 例如，对于一个特征A，如果它有k个取值，那么固有值的计算公式为：IV(A) = -∑(P(v) * log2(P(v)))，其中P(v)是特征A取值v的概率。
 - 注意,公式计算看起来与熵的计算类似,但是区别：
   - 熵是分别对特征中的**每个取值的子集对Target**进行计算概率和熵，然后再计算加权平均熵
   - 但是固有值计算是**直接对特征的所有取值**进行概率计算，**而不是针对Target**，所以叫做特征的固有值。
3. **计算信息增益率**：将信息增益除以特征的固有值，得到信息增益率。
4. **选择最优特征**：选择信息增益率最高的特征作为划分特征。

#### 3. C4.5决策树的手算方法

##### 3.1 数据准备

In [1]:
import pandas as pd
# 创建数据集
data = {
    "Weather": ["Sunny", "Sunny", "Overcast", "Rainy", "Rainy", "Rainy", "Overcast", "Sunny", "Sunny", "Rainy"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool", "Cool", "Mild", "Mild", "Cool", "Mild"],
    "Humidity": ["High", "High", "High", "High", "Normal", "Normal", "Normal", "High", "Normal", "Normal"],
    "Windy": ["False", "True", "False", "False", "False", "True", "True", "False", "False", "False"],
    "Play": ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes"]
}
df = pd.DataFrame(data)
df

,Weather,Temperature,Humidity,Windy,Play
0,Sunny,Hot,High,False,No
1,Sunny,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Rainy,Mild,High,False,Yes
4,Rainy,Cool,Normal,False,Yes
5,Rainy,Cool,Normal,True,No
6,Overcast,Mild,Normal,True,Yes
7,Sunny,Mild,High,False,No
8,Sunny,Cool,Normal,False,Yes
9,Rainy,Mild,Normal,False,Yes


##### 3.2 对于根节点的特征选择
在根节点上，我们需要选择一个最优特征来划分数据集。我们将计算每个特征的信息增益，并选择信息增益最大的特征作为根节点的划分特征。

In [2]:
# 计算根节点的熵
# 当前根结点中一共有10个样本，其中Yes为6个，No为4个
from math import log2
H_root = -(6/10)*log2(6/10) - (4/10)*log2(4/10)
H_root

0.9709505944546686

###### 3.2.1 计算特征Weather的信息增益率

In [3]:
# 对于特征Weather，我们需要计算每个取值的子集的熵，并计算加权平均熵
# Sunny子集： 一共有5个样本，其中Yes为2个，No为3个
H_sunny = -(2/5)*log2(2/5) - (3/5)*log2(3/5)
# Overcast子集： 一共2个样本，其中Yes为2个，No为0个
H_overcast = -(2/2)*log2(2/2)  # 注意：当概率为0时，熵为0
# Rainy子集： 一共3个样本， 其中Yes为2个，No为1个
H_rainy = -(2/3)*log2(2/3) - (1/3)*log2(1/3)
# 计算加权平均熵
H_weather = (5/10)*H_sunny + (2/10)*H_overcast + (3/10)*H_rainy
# 计算信息增益
IG_weather = H_root - H_weather

# 计算特征Weather的固有值
P_weather = -(5/10)*log2(5/10) - (2/10)*log2(2/10) - (3/10)*log2(3/10)
# 计算信息增益率
IGR_weather = IG_weather / P_weather
IGR_weather

0.14135983775895228

###### 3.2.2 计算特征Temperature的信息增益

In [6]:
# 对于特征Temperature，我们需要计算每个取值的子集的熵，并计算加权平均熵
# Hot子集： 一共有3个样本，其中Yes为1个，No为2个
H_hot = -(1/3)*log2(1/3) - (2/3)*log2(2/3)
# Mild子集： 一共4个样本，其中Yes为3个， No为1个
H_mild = -(3/4)*log2(3/4) - (1/4)*log2(1/4)
# Cool子集： 一共3个样本，其中Yes为2个， No为1个
H_cool = -(2/3)*log2(2/3) - (1/3)*log2(1/3)
# 计算加权平均熵
H_temperature = (3/10)*H_hot + (4/10)*H_mild + (3/10)*H_cool
# 计算信息增益
IG_temperature = H_root - H_temperature

# 计算特征Temperature的固有值
P_temperature = -(3/10)*log2(3/10) - (4/10)*log2(4/10) - (3/10)*log2(3/10)
# 计算信息增益率
IGR_temperature = IG_temperature / P_temperature
IGR_temperature

0.060766929638204084

###### 3.2.3 计算特征Humidity的信息增益


In [4]:
# 对于特征Humidity， 我们需要计算每个取值的子集的熵，并计算加权平均熵
# High子集，一共有5个样本，其中Yes有2个，No有3个
H_high = -(2/5)*log2(2/5) - (3/5)*log2(3/5)
# Normal子集，一共有5个样本，其中Yes有4个， No有1个
H_normal = -(4/5)*log2(4/5) - (1/5)*log2(1/5)
# 计算加权平均熵
H_humidity = (5/10)*H_high + (5/10)*H_normal
# 计算信息增益
IG_humidity = H_root - H_humidity

# 计算 Humidity 的固有值
P_humidify = -(5/10)*log2(5/10) - (5/10)*log2(5/10)
# 计算信息增益率
IGR_humidity = IG_humidity / P_humidify
IGR_humidity

0.12451124978365313

###### 3.2.4 计算特征Windy的信息增益

In [5]:
# 对于特征Windy，我们需要计算每个取值的子集的熵，并计算加权平均熵
# False子集，一共有7个样本，其中Yes有5个， No有2个
H_false = -(5/7)*log2(5/7) - (2/7)*log2(2/7)
# True子集，一共有3个样本，其中Yes有1个， No有2个
H_true = -(1/3)*log2(1/3) - (2/3)*log2(2/3)
# 计算加权平均熵
H_windy = (7/10)*H_false + (3/10)*H_true
# 计算信息增益
IG_windy = H_root - H_windy

# 计算 Windy 的固有值
P_windy = -(7/10)*log2(7/10) - (3/10)*log2(3/10)
# 计算信息增益率
IGR_windy = IG_windy / P_windy
IGR_windy

0.10357243711623386

###### 3.2.5 选择最优特征进行划分
对比每个特征的信息增益率，我们可以发现特征Weather的信息增益率最大，因此我们选择特征Weather作为根节点的划分特征。

In [9]:
max_IGR = max(IGR_weather, IGR_temperature, IGR_humidity, IGR_windy)
max_IGR

0.14135983775895228

#### 4. 分裂后得到的结果
通过对根节点进行划分，原始数据被拆分为三个子节点，分别对应特征Weather的取值:Sunny、Overcast和Rainy。 <br>
 - 对于Sunny子节点，我们需要继续计算特征Temperature、Humidity和Windy的信息增益，并选择信息增益最大的特征进行划分。 <br>
 - 对于Overcast子节点，由于所有样本的目标变量都是Yes，因此我们可以直接将Overcast子节点标记为叶子节点，类别为Yes。 <br>
 - 对于Rainy子节点，我们需要继续计算特征Temperature、Humidity和Windy的信息增益，并选择信息增益最大的特征进行划分。 <br>
通过递归地选择最优特征进行划分，我们能够逐步构建出一个树形结构，从而实现对数据的分类。 <br>
通过以上步骤，我们可以构建出一个完整的ID3决策树，从而实现对数据的分类。 <br>
